# Método 1: 🛰️ Processamento de Dados InSAR — Integração ASC/DESC e ORTHO

Este notebook realiza o processamento de séries temporais InSAR provenientes de órbitas **ascendentes (ASC)** e **descendentes (DESC)**, com o objetivo de obter as componentes **vertical (dV)** e **horizontal (dH)** do deslocamento do terreno, e compará-las com as medições **ortogonais (ORTHO)**.

---

## Etapas principais

### 1. Leitura e filtragem dos dados
- Leitura dos ficheiros CSV de **ASC**, **DESC** e **ORTHO**.  
- Filtragem da **área de interesse** com base nas coordenadas (northing/easting).

### 2. Transformação temporal e interpolação
- Conversão dos dados para **formato longo** (`melt`) para facilitar a manipulação temporal.  
- **Interpolação temporal linear** para alinhar as datas comuns entre as órbitas ASC e DESC.

---

## Interpolação Espacial — IDW (Inverse Distance Weighting)

O método **IDW** estima o valor de uma variável num ponto alvo com base nos valores conhecidos dos pontos vizinhos, **ponderados inversamente à distância**.

Para cada data:
1. Cria-se uma **árvore espacial** (`cKDTree`) com os pontos de origem.  
2. Para cada ponto alvo, identificam-se os vizinhos dentro de um **raio definido** (ex. 150 m).  
3. Calcula-se o valor interpolado como média ponderada:

$$
\hat{z}(x) = \frac{\sum_i w_i z_i}{\sum_i w_i}, \quad w_i = \frac{1}{d_i^p}
$$

---

### Cálculo da distância \(d_i\) e ponderação

- A distância \( d_i \) é **euclidiana** no plano das coordenadas Easting/Northing:

$$
d_i = \sqrt{(x_t - x_i)^2 + (y_t - y_i)^2}
$$

onde:
- \( (x_t, y_t) \) → coordenadas do ponto alvo (onde queremos interpolar)  
- \( (x_i, y_i) \) → coordenadas do ponto vizinho de origem  

- Parâmetros usados no script:
  - **Raio máximo (`radius`)**: 150 m — vizinhos mais distantes são ignorados  
  - **Número de vizinhos (`k`)**: 5 — máximo de vizinhos utilizados na média ponderada  
  - **Potência (`power`)**: 2 — o peso dos vizinhos é inversamente proporcional ao quadrado da distância:

$$
w_i = \frac{1}{d_i^2}
$$

Isto garante que **pontos próximos têm muito mais influência** que pontos distantes.

---

## Cálculo das Componentes Verticais e Horizontais (dV e dH)

A partir dos deslocamentos **na linha de visada (LOS)** das órbitas ASC e DESC, calculam-se as componentes **vertical (dV)** e **horizontal (dH)** reais do movimento do terreno.

$$
dV = \frac{d_{DESC}\sin\theta_A\cos(\beta-\gamma) + d_{ASC}\sin\theta_D\cos(\beta+\gamma)}{D}
$$

$$
dH = \frac{d_{DESC}\cos\theta_A - d_{ASC}\cos\theta_D}{D}
$$

onde:  
- $\theta_A, \theta_D$ — ângulos de incidência ASC/DESC  
- $\alpha_A, \alpha_D$ — ângulos de trajetória  
- $\beta, \gamma$ — latitude e orientação orbital  
- $D$ — denominador da combinação geométrica  

Estas fórmulas permitem decompor os deslocamentos LOS em componentes físicas **verticais e horizontais**.

---

## Geração de Grelha e Visualização

- Cria-se uma **grelha regular (ex. 100 m)** centrada nos dados ORTHO.  
- Calculam-se as **médias por célula e data** para dV e dH.  
- Produzem-se:
  - Um **mapa espacial** das células selecionadas.  
  - **Séries temporais comparativas** ASC/DESC vs ORTHO (mesma escala).

---

## Resultado

O processo permite:
- Obter deslocamentos **verticais e horizontais coerentes**.  
- Corrigir **discrepâncias espaciais** entre órbitas ASC/DESC.  
- **Comparar diretamente** com medições ortogonais independentes (ORTHO).  
- Fornecer uma base sólida para análise de deformações do terreno.

---

# Cálculo de deslocamentos asc/desc + ortho para células selecionadas

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Células selecionadas
# ==============================
selected_ids = ["3_5","4_5","5_5","6_5","7_5",
                "3_4","4_4","5_4","6_4",
                "5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids)]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]

# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos para cada célula (com mesma escala)
# ==============================

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_sel['dV'].min(),
    agg_sel['dH'].min(),
    ortho_v_sel['disp'].min() if not ortho_v_sel.empty else np.inf,
    ortho_h_sel['disp'].min() if not ortho_h_sel.empty else np.inf
)
v_max = max(
    agg_sel['dV'].max(),
    agg_sel['dH'].max(),
    ortho_v_sel['disp'].max() if not ortho_v_sel.empty else -np.inf,
    ortho_h_sel['disp'].max() if not ortho_h_sel.empty else -np.inf
)

# Número de células
n_cells = len(selected_ids)
n_cols = 4
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*4), sharex=True, sharey=True)

# Garantir que axes é 2D
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids):
    row_idx = i // n_cols
    col_idx = i % n_cols

    ax = axes[row_idx, col_idx]

    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # dV
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')

    # dH
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f'Célula {cid}', fontsize=10)
    ax.set_ylabel('Deslocamento (mm)')
    ax.set_xlabel('Data')
    ax.legend(fontsize=7)
    ax.set_ylim(v_min, v_max)  # Escala comum

# Remove eixos extras se houver células a menos que a grid
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file)
desc = pd.read_csv(desc_file)
ortho_v = pd.read_csv(ortho_v_file)
ortho_h = pd.read_csv(ortho_h_file)

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Células selecionadas
# ==============================
selected_ids = ["2_5","3_5","4_5","5_5","6_5","7_5",
                "2_4","3_4","4_4","5_4","6_4","7_4",
                "2_6","3_6","4_6","5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids)]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]

# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos para cada célula (com mesma escala)
# ==============================

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_sel['dV'].min(),
    agg_sel['dH'].min(),
    ortho_v_sel['disp'].min() if not ortho_v_sel.empty else np.inf,
    ortho_h_sel['disp'].min() if not ortho_h_sel.empty else np.inf
)
v_max = max(
    agg_sel['dV'].max(),
    agg_sel['dH'].max(),
    ortho_v_sel['disp'].max() if not ortho_v_sel.empty else -np.inf,
    ortho_h_sel['disp'].max() if not ortho_h_sel.empty else -np.inf
)

# Número de células
n_cells = len(selected_ids)
n_cols = 2
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*4), sharex=True, sharey=True)

# Garantir que axes é 2D
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids):
    row_idx = i // n_cols
    col_idx = i % n_cols

    ax = axes[row_idx, col_idx]

    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # dV
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')

    # dH
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f'Célula {cid}', fontsize=10)
    ax.set_ylabel('Deslocamento (mm)')
    ax.set_xlabel('Data')
    ax.legend(fontsize=7)
    ax.set_ylim(v_min, v_max)  # Escala comum

# Remove eixos extras se houver células a menos que a grid
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file, engine='python')
desc = pd.read_csv(desc_file, engine='python')
ortho_v = pd.read_csv(ortho_v_file, engine='python')
ortho_h = pd.read_csv(ortho_h_file, engine='python')

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Células selecionadas
# ==============================
selected_ids = ["2_5","3_5","4_5","5_5","6_5","7_5",
                "2_4","3_4","4_4","5_4","6_4","7_4",
                "2_6","3_6","4_6","5_6","6_6","7_6"]
agg_sel = agg[agg['cell_id'].isin(selected_ids)]
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]

# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_long = melt_ortho(ortho_v)
ortho_h_long = melt_ortho(ortho_h)

# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos para cada célula (com mesma escala)
# ==============================

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_sel['dV'].min(),
    agg_sel['dH'].min(),
    ortho_v_sel['disp'].min() if not ortho_v_sel.empty else np.inf,
    ortho_h_sel['disp'].min() if not ortho_h_sel.empty else np.inf
)
v_max = max(
    agg_sel['dV'].max(),
    agg_sel['dH'].max(),
    ortho_v_sel['disp'].max() if not ortho_v_sel.empty else -np.inf,
    ortho_h_sel['disp'].max() if not ortho_h_sel.empty else -np.inf
)

# Número de células
n_cells = len(selected_ids)
n_cols = 3
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows*4), sharex=True, sharey=True)

# Garantir que axes é 2D
if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(selected_ids):
    row_idx = i // n_cols
    col_idx = i % n_cols

    ax = axes[row_idx, col_idx]

    agg_cell = agg_sel[agg_sel['cell_id'] == cid]
    ortho_v_cell = ortho_v_sel[ortho_v_sel['cell_id'] == cid]
    ortho_h_cell = ortho_h_sel[ortho_h_sel['cell_id'] == cid]

    # dV
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')

    # dH
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f'Célula {cid}', fontsize=10)
    ax.set_ylabel('Deslocamento (mm)')
    ax.set_xlabel('Data')
    ax.legend(fontsize=7)
    ax.set_ylim(v_min, v_max)  # Escala comum

# Remove eixos extras se houver células a menos que a grid
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

plt.tight_layout()
plt.show()

# Cálculo de deslocamentos asc/desc + ortho para todas as células

In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import contextily as ctx
import numpy as np
from scipy.spatial import cKDTree

# ==============================
# 1. Ler CSVs ASC, DESC e ORTHO
# ==============================
asc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_147_0224_IW2_VV_2019_2023_1/EGMS_L2b_147_0224_IW2_VV_2019_2023_1.csv"
desc_file = "data/alqueva_calibrated_asc_desc_2019_2023/EGMS_L2b_052_0848_IW2_VV_2019_2023_1/EGMS_L2b_052_0848_IW2_VV_2019_2023_1.csv"
ortho_v_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_U_2019_2023_1/EGMS_L3_E27N18_100km_U_2019_2023_1.csv"
ortho_h_file = "data/alqueva_calibrated_ortho_2019_2023/EGMS_L3_E27N18_100km_E_2019_2023_1/EGMS_L3_E27N18_100km_E_2019_2023_1.csv"

asc = pd.read_csv(asc_file, engine='python')
desc = pd.read_csv(desc_file, engine='python')
ortho_v = pd.read_csv(ortho_v_file, engine='python')
ortho_h = pd.read_csv(ortho_h_file, engine='python')

# ==============================
# 2. Filtrar área de interesse
# ==============================
norte_min, norte_max = 1855050, 1855850
este_min, este_max = 2792250, 2793250

asc = asc[(asc['northing'] >= norte_min) & (asc['northing'] <= norte_max) &
          (asc['easting'] >= este_min) & (asc['easting'] <= este_max)]
desc = desc[(desc['northing'] >= norte_min) & (desc['northing'] <= norte_max) &
            (desc['easting'] >= este_min) & (desc['easting'] <= este_max)]
ortho_v = ortho_v[(ortho_v['northing'] >= norte_min) & (ortho_v['northing'] <= norte_max) &
                  (ortho_v['easting'] >= este_min) & (ortho_v['easting'] <= este_max)]
ortho_h = ortho_h[(ortho_h['northing'] >= norte_min) & (ortho_h['northing'] <= norte_max) &
                  (ortho_h['easting'] >= este_min) & (ortho_h['easting'] <= este_max)]

# ==============================
# 3. Função melt para ASC/DESC
# ==============================
def melt_to_long(df):
    disp_cols = df.columns[24:]  # Ajustar conforme as colunas de datas
    long_df = df.melt(
        id_vars=['easting','northing','incidence_angle','track_angle','latitude','longitude'],
        value_vars=disp_cols, var_name='date', value_name='disp'
    )
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

asc_long = melt_to_long(asc)
desc_long = melt_to_long(desc)

common_dates = pd.date_range(
    start=max(asc_long['date'].min(), desc_long['date'].min()),
    end=min(asc_long['date'].max(), desc_long['date'].max()),
    freq='MS'
)

# ==============================
# 4. Interpolação temporal linear
# ==============================
def interpolate_ps(df, common_dates):
    dfs = []
    for (x, y), group in df.groupby(['easting','northing']):
        group = group.sort_values('date')
        interp = np.interp(
            pd.to_datetime(common_dates).astype(np.int64),
            group['date'].astype(np.int64),
            group['disp']
        )
        dfs.append(pd.DataFrame({
            'easting': x,
            'northing': y,
            'latitude': group['latitude'].iloc[0],
            'longitude': group['longitude'].iloc[0],
            'date': common_dates,
            'disp': interp,
            'incidence_angle': group['incidence_angle'].iloc[0],
            'track_angle': group['track_angle'].iloc[0]
        }))
    return pd.concat(dfs, ignore_index=True)

asc_interp = interpolate_ps(asc_long, common_dates)
desc_interp = interpolate_ps(desc_long, common_dates)

# ==============================
# 5. Interpolação espacial IDW
# ==============================
def idw_interpolation_per_date(source_df, target_df, radius=150, power=2):
    out_list = []
    for date, src_group in source_df.groupby('date'):
        trg_group = target_df[target_df['date']==date].copy()
        if src_group.empty or trg_group.empty:
            continue
        src_points = np.array(list(zip(src_group['easting'], src_group['northing'])))
        trg_points = np.array(list(zip(trg_group['easting'], trg_group['northing'])))
        tree = cKDTree(src_points)
        dists, idxs = tree.query(trg_points, k=5, distance_upper_bound=radius)

        interpolated_disp = []
        interpolated_theta = []
        interpolated_alpha = []
        for dist, idx in zip(dists, idxs):
            mask = np.isfinite(dist)
            if not np.any(mask):
                interpolated_disp.append(np.nan)
                interpolated_theta.append(np.nan)
                interpolated_alpha.append(np.nan)
                continue
            weights = 1 / (dist[mask] ** power)
            interpolated_disp.append(np.sum(weights * src_group.iloc[idx[mask]]['disp']) / np.sum(weights))
            interpolated_theta.append(np.sum(weights * src_group.iloc[idx[mask]]['incidence_angle']) / np.sum(weights))
            interpolated_alpha.append(np.sum(weights * src_group.iloc[idx[mask]]['track_angle']) / np.sum(weights))

        trg_group['disp_idw'] = interpolated_disp
        trg_group['theta_desc'] = interpolated_theta
        trg_group['alpha_desc'] = interpolated_alpha
        out_list.append(trg_group)
    return pd.concat(out_list, ignore_index=True)

asc_interp = idw_interpolation_per_date(desc_interp, asc_interp)

# ==============================
# 6. Calcular β e γ
# ==============================
orbit_inclination = np.deg2rad(98.6)
asc_interp['beta'] = np.arcsin(np.cos(orbit_inclination) * np.cos(np.deg2rad(asc_interp['latitude'])))
asc_interp['gamma'] = 0

# ==============================
# 7. Calcular dV e dH
# ==============================
def compute_dV_dH_real(row):
    θA = np.deg2rad(row['incidence_angle'])
    θD = np.deg2rad(row['theta_desc'])
    αA = np.deg2rad(row['track_angle'])
    αD = np.deg2rad(row['alpha_desc'])
    β = row['beta']
    γ = row['gamma']

    dASC_LOS = row['disp']
    dDESC_LOS = row['disp_idw']

    denom = (np.cos(θA)*np.sin(θD)*np.cos(β + γ) +
             np.cos(θD)*np.sin(θA)*np.cos(β - γ))

    dV = (dDESC_LOS*np.sin(θA)*np.cos(β - γ) +
          dASC_LOS*np.sin(θD)*np.cos(β + γ)) / denom
    dH = (dDESC_LOS*np.cos(θA) - dASC_LOS*np.cos(θD)) / denom
    return pd.Series({'dV': dV, 'dH': dH})

asc_interp[['dV','dH']] = asc_interp.apply(compute_dV_dH_real, axis=1)

# ==============================
# 8. Grelha centrada ORTHO
# ==============================
grid_size = 100
x_edges = np.arange(asc_interp['easting'].min(), asc_interp['easting'].max()+grid_size, grid_size)
y_edges = np.arange(asc_interp['northing'].min(), asc_interp['northing'].max()+grid_size, grid_size)
x_edges_shifted = x_edges - grid_size/2
y_edges_shifted = y_edges - grid_size/2

asc_interp['cell_x'] = pd.cut(asc_interp['easting'], bins=x_edges_shifted, labels=False)
asc_interp['cell_y'] = pd.cut(asc_interp['northing'], bins=y_edges_shifted, labels=False)

agg = asc_interp.dropna(subset=['cell_x','cell_y']).groupby(['cell_x','cell_y','date']).agg(
    x_center=('easting','mean'),
    y_center=('northing','mean'),
    dV=('dV','mean'),
    dH=('dH','mean')
).reset_index()
agg['cell_id'] = agg['cell_x'].astype(int).astype(str) + "_" + agg['cell_y'].astype(int).astype(str)

# ==============================
# 9. GeoDataFrames
# ==============================
grid_data = []
for ix in range(len(x_edges_shifted)-1):
    for iy in range(len(y_edges_shifted)-1):
        grid_data.append({
            "cell_x": ix,
            "cell_y": iy,
            "cell_id": f"{ix}_{iy}",
            "geometry": box(x_edges_shifted[ix], y_edges_shifted[iy],
                            x_edges_shifted[ix+1], y_edges_shifted[iy+1])
        })
grid = gpd.GeoDataFrame(grid_data, crs="EPSG:3035").to_crs(epsg=3857)
points = agg.groupby('cell_id').first().reset_index()
gdf_points = gpd.GeoDataFrame(
    points,
    geometry=gpd.points_from_xy(points['x_center'], points['y_center']),
    crs="EPSG:3035"
).to_crs(epsg=3857)

# ==============================
# 10. Usar TODAS as células com dados
# ==============================
selected_ids = agg['cell_id'].unique().tolist()
agg_sel = agg.copy()
grid_sel = grid[grid['cell_id'].isin(selected_ids)]
points_sel = gdf_points[gdf_points['cell_id'].isin(selected_ids)]



# ==============================
# 11. ORTHO em formato longo e atribuição de células
# ==============================
def melt_ortho(df):
    disp_cols = df.columns[24:]  # ajustar conforme datas ORTHO
    long_df = df.melt(id_vars=['easting','northing'], value_vars=disp_cols,
                      var_name='date', value_name='disp')
    long_df['date'] = pd.to_datetime(long_df['date'], errors='coerce')
    return long_df.dropna(subset=['disp','date'])

ortho_v_sel = ortho_v_long.copy()
ortho_h_sel = ortho_h_long.copy()


# atribuir células
for ortho_df in [ortho_v_long, ortho_h_long]:
    ortho_df['cell_x'] = pd.cut(ortho_df['easting'], bins=x_edges_shifted, labels=False)
    ortho_df['cell_y'] = pd.cut(ortho_df['northing'], bins=y_edges_shifted, labels=False)
    ortho_df.dropna(subset=['cell_x','cell_y'], inplace=True)
    ortho_df['cell_id'] = ortho_df['cell_x'].astype(int).astype(str) + "_" + ortho_df['cell_y'].astype(int).astype(str)

ortho_v_sel = ortho_v_long[ortho_v_long['cell_id'].isin(selected_ids)]
ortho_h_sel = ortho_h_long[ortho_h_long['cell_id'].isin(selected_ids)]

# ==============================
# 12. Mapa das células selecionadas + pontos ORTHO (com legenda organizada)
# ==============================

# Criar figura
fig_map, ax_map = plt.subplots(figsize=(10, 10))

# Grelha base
grid.boundary.plot(ax=ax_map, color='lightgray', linewidth=0.5, 
                   label=f'Grelha Base ({grid_size} m)')

# Células selecionadas
grid_sel.boundary.plot(ax=ax_map, color='red', linewidth=1.5, 
                       label=f'Células Selecionadas ({grid_size} m)')

# Pontos ASC/DESC (centros das células)
points_sel.plot(ax=ax_map, color='white', edgecolor='black', markersize=60, label='Pontos ASC/DESC')

# Criar GeoDataFrame combinado dos pontos ORTHO selecionados
ortho_comb = pd.concat([ortho_v_sel, ortho_h_sel], ignore_index=True)

# Evitar duplicados exatos
ortho_comb = ortho_comb.drop_duplicates(subset=['easting', 'northing', 'date'])

# Transformar em GeoDataFrame
gdf_ortho_comb = gpd.GeoDataFrame(
    ortho_comb,
    geometry=gpd.points_from_xy(ortho_comb['easting'], ortho_comb['northing']),
    crs="EPSG:3035"
).to_crs(epsg=3857)


# Pontos ORTHO combinados (preto)
if not gdf_ortho_comb.empty:
    gdf_ortho_comb.plot(ax=ax_map, color='black', markersize=25, alpha=0.8, label='Pontos ORTHO')

# IDs das células
for _, row in grid_sel.iterrows():
    x_min, y_min, x_max, y_max = row['geometry'].bounds
    ax_map.text(
        x_min, y_max, row['cell_id'],
        fontsize=9, ha='left', va='top', color='black',
        bbox=dict(facecolor='white', alpha=0.6, pad=1)
    )

# Adicionar basemap e formato
ctx.add_basemap(ax_map, source=ctx.providers.Esri.WorldImagery)
ax_map.set_title("Mapa das Células Selecionadas com Pontos ASC/DESC e ORTHO", fontsize=14)
ax_map.set_axis_off()

# Legenda organizada (inclui o tamanho da grelha)
ax_map.legend(loc='lower left', fontsize=9, frameon=True)

plt.show()


# ==============================
# 13. Grid de gráficos - TODAS AS CÉLULAS
# ==============================

# Usar todas as células disponíveis dentro dos limites
agg_all = agg.copy()

# Calcular limites globais (min e max de todos os dados)
v_min = min(
    agg_all['dV'].min(),
    agg_all['dH'].min(),
    ortho_v_long['disp'].min() if not ortho_v_long.empty else np.inf,
    ortho_h_long['disp'].min() if not ortho_h_long.empty else np.inf
)
v_max = max(
    agg_all['dV'].max(),
    agg_all['dH'].max(),
    ortho_v_long['disp'].max() if not ortho_v_long.empty else -np.inf,
    ortho_h_long['disp'].max() if not ortho_h_long.empty else -np.inf
)

# Todas as células (ordenadas por Y e X)
all_ids = sorted(agg_all['cell_id'].unique(), key=lambda s: (int(s.split('_')[1]), int(s.split('_')[0])))

# Número de células
n_cells = len(all_ids)
n_cols = 3  # Ajusta se quiseres mais/menos colunas
n_rows = int(np.ceil(n_cells / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows*4), sharex=True, sharey=True)

if n_rows == 1:
    axes = axes.reshape(1, -1)

for i, cid in enumerate(all_ids):
    r = i // n_cols
    c = i % n_cols
    ax = axes[r, c]

    agg_cell = agg_all[agg_all['cell_id'] == cid]
    ortho_v_cell = ortho_v_long[ortho_v_long['cell_id'] == cid]
    ortho_h_cell = ortho_h_long[ortho_h_long['cell_id'] == cid]

    # ASC/DESC
    ax.plot(agg_cell['date'], agg_cell['dV'], color='blue', label='ASC/DESC dV')
    ax.plot(agg_cell['date'], agg_cell['dH'], color='green', label='ASC/DESC dH')

    # ORTHO (tracejado)
    if not ortho_v_cell.empty:
        ax.plot(ortho_v_cell['date'], ortho_v_cell['disp'], color='blue', linestyle='--', label='ORTHO dV')
    if not ortho_h_cell.empty:
        ax.plot(ortho_h_cell['date'], ortho_h_cell['disp'], color='green', linestyle='--', label='ORTHO dH')

    ax.set_title(f"Célula {cid}", fontsize=9)
    ax.set_ylabel("Deslocamento (mm)")
    ax.set_xlabel("Data")
    ax.set_ylim(v_min, v_max)
    ax.legend(fontsize=7, loc='best')

# Remove eixos vazios
for j in range(i+1, n_rows*n_cols):
    r = j // n_cols
    c = j % n_cols
    fig.delaxes(axes[r, c])

#plt.suptitle("Séries Temporais - Todas as Células na Área de Interesse", fontsize=14)
plt.tight_layout()
plt.show()

# Método 2